In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, EarlyStoppingCallback

In [ ]:
dataset = load_dataset("AbstractTTS/SAVEE", split="train")
print(dataset[0])
print(dataset.features)

In [ ]:
def load_savee_speaker_independent():
    dataset = load_dataset("AbstractTTS/SAVEE", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        return example["file"].split("_")[0]

    # 4 ομιλητές: DC, JE, JK, KL
    train_speakers = ["DC", "JE"]
    test_speakers = ["JK", "KL"]

    train_dataset = dataset.filter(lambda x: get_speaker(x) in train_speakers)
    test_dataset = dataset.filter(lambda x: get_speaker(x) in test_speakers)

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_savee_speaker_independent()

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))
print(labels)

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

#υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
model_repo = "facebook/hubert-base-ls960"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_repo)
model = AutoModelForAudioClassification.from_pretrained(model_repo,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")
model.freeze_feature_encoder()

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = label2id[batch["emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

max_abs_vals = [np.max(np.abs(encoded_dataset["train"][i]["input_values"])) for i in range(len(encoded_dataset["train"]))]
print("Μέγιστη απόλυτη τιμή (train):", max(max_abs_vals))

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    average_precision_score, log_loss, confusion_matrix,
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    cm = confusion_matrix(labels_true, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

In [ ]:
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
# Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 15

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

training_args = TrainingArguments(
    output_dir="./hubert-finetuned-SAVEE",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    logging_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/hubert-finetuned-SAVEE"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")